# NeuroSense AI: stereo taqqoslash
GPU tanlang va Run all bosing. Middlebury 2021 chess1 va skates1: ikkala xaritada bir xil shkala. MAE pikselda; bad2_fraction — xatosi 2 pikseldan katta ulush. Natijalar umumiy yaroqli piksellarda hisoblanadi.

In [ ]:
import sys, subprocess, shutil, json, urllib.request
from pathlib import Path
import torch, cv2
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU tanlang, keyin Run all.")
def run(*args, cwd=None):
    subprocess.run([str(a) for a in args], cwd=cwd, check=True)
root = Path('/content/neurosense-study')
root.mkdir(exist_ok=True)
raft = root/'RAFT-Stereo'
if not raft.exists():
    run('git','clone','--depth','1','https://github.com/princeton-vl/RAFT-Stereo.git',raft)
print(torch.cuda.get_device_name(), torch.__version__, cv2.__version__)
run('git','rev-parse','HEAD',cwd=raft)
run(sys.executable,'-m','pip','-q','install','gdown')
import gdown
ckpt = root/'raftstereo-middlebury.pth'
if not ckpt.exists():
    tmp = ckpt.with_suffix('.partial')
    if not gdown.download(id='1m3KoukUmKDoMv-ySOO6vBzYfWLyj9yqd', output=str(tmp), quiet=False):
        raise RuntimeError('Model yuklanmadi; yuklash katagini qayta ishlating.')
    tmp.replace(ckpt)
url = 'https://raw.githubusercontent.com/Ikboloy/NeuroSense-AINeuroSense-AI/bc1a8281c20df59820af59e4b3848b4198856065/experiments/raft-stereo/compare.py'
urllib.request.urlretrieve(url, root/'compare.py')

In [ ]:
from IPython.display import display, Image
import zipfile
from PIL import Image as PILImage
from google.colab import files
archive = Path('/content/neurosense-scenes.zip')
if not archive.is_file():
    print('Upload neurosense-scenes.zip (chess1/ and skates1/ folders).')
    files.upload()
out = root/'results'
for scene in ('chess1','skates1'):
    folder = root/scene
    folder.mkdir(exist_ok=True)
    for name in ('im0.png','im1.png','calib.txt','disp0.pfm'):
        with zipfile.ZipFile(archive) as z:
            (folder/name).write_bytes(z.read(f'{scene}/{name}'))
        if name.endswith('.png'):
            with PILImage.open(folder/name) as image:
                rgb = image.convert('RGB')
            rgb.save(folder/name)
    flow = root/'flows'
    run(sys.executable,'demo.py','--restore_ckpt',ckpt,'--corr_implementation','alt','--mixed_precision','--save_numpy','-l',folder/'im0.png','-r',folder/'im1.png','--output_directory',flow,cwd=raft)
    run(sys.executable,root/'compare.py',folder,flow/(scene+'.npy'),out/scene)
    print(scene, (out/scene/'metrics.json').read_text())
    print('Chap surat | SGBM | RAFT-Stereo')
    display(Image(filename=str(out/scene/'left-sgbm-raft.png'),width=1200))

In [ ]:
from google.colab import files
files.download(shutil.make_archive(str(root/'NeuroSense-results'),'zip',out))